# DeepSight Pro — Deepfake Detection from Scratch (No Pretrained Weights)

This notebook builds and trains a custom CNN **from random initialization**. It does not download or use ImageNet weights, pretrained checkpoints, or transfer learning.

The workflow includes:
1. Dataset inventory and metadata validation
2. Corrupt-image checks and exact duplicate detection
3. Source/generator-group-disjoint train, validation and test splits
4. Image preprocessing and training-only augmentation
5. Class-imbalance handling
6. Custom CNN architecture built from scratch
7. Training, learning-rate scheduling and checkpointing
8. Validation-based threshold selection
9. Independent test evaluation and confusion matrix
10. Compression stress test, Grad-CAM-style visualization and model export

## Dataset format

Create `dataset/metadata.csv` with columns:

| filepath | label | group_id |
|---|---|---|
| images/real/a001.jpg | REAL | real_source_01 |
| images/fake/b001.jpg | FAKE | generator_batch_01 |

- `filepath`: path relative to `dataset/` or absolute path.
- `label`: exactly `REAL` or `FAKE`.
- `group_id`: original source/video/person/generator batch.
- Put related crops, frames and near-duplicates in the same group.

**Dataset quality matters more than model complexity.** Do not mix the same source, video, identity, generator batch or near-duplicate images across splits. A group split cannot fix inaccurate group IDs.

In [ ]:
%pip install -q tensorflow pandas scikit-learn matplotlib pillow seaborn

In [ ]:
from pathlib import Path
from collections import Counter
from io import BytesIO
import hashlib, os, random, json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, precision_recall_curve, f1_score, accuracy_score,
    precision_score, recall_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
ImageFile.LOAD_TRUNCATED_IMAGES = False

DATA_DIR = Path("dataset")
MANIFEST = DATA_DIR / "metadata.csv"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 35
AUTOTUNE = tf.data.AUTOTUNE
CLASS_TO_ID = {"FAKE": 0, "REAL": 1}
ID_TO_CLASS = {v:k for k,v in CLASS_TO_ID.items()}

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
assert MANIFEST.exists(), "Create dataset/metadata.csv first."

## 1. Load metadata and inspect the dataset

In [ ]:
df = pd.read_csv(MANIFEST)
required = {"filepath", "label", "group_id"}
missing = required - set(df.columns)
assert not missing, f"Missing columns: {missing}"

df["label"] = df["label"].astype(str).str.upper().str.strip()
df["group_id"] = df["group_id"].astype(str).str.strip()
assert set(df["label"].unique()) == {"FAKE", "REAL"}, "Labels must be exactly FAKE and REAL."
assert df["group_id"].ne("").all(), "Every image must have a group_id."

def resolve_path(value):
    p = Path(str(value))
    return p if p.is_absolute() else DATA_DIR / p

df["full_path"] = df["filepath"].map(resolve_path)
missing_paths = df[~df["full_path"].map(Path.exists)]
print("Missing image paths:", len(missing_paths))
df = df[df["full_path"].map(Path.exists)].copy()
assert len(df), "No valid image paths found."

print("Total images:", len(df))
print("Class distribution:\n", df["label"].value_counts())
print("Unique source/generator groups:", df["group_id"].nunique())
print("Groups per class:\n", df.groupby("label")["group_id"].nunique())

## 2. Validate image files and remove unreadable samples

In [ ]:
valid_rows, corrupt_rows = [], []
for _, row in df.iterrows():
    try:
        with Image.open(row["full_path"]) as im:
            im.verify()
        valid_rows.append(row)
    except Exception as exc:
        corrupt_rows.append({"filepath": str(row["full_path"]), "error": str(exc)})

df = pd.DataFrame(valid_rows).reset_index(drop=True)
print("Valid images:", len(df), "| Corrupt images:", len(corrupt_rows))
if corrupt_rows:
    pd.DataFrame(corrupt_rows).to_csv("corrupt_images_report.csv", index=False)
assert len(df), "No readable images remain."

## 3. Exact duplicate detection

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

df["sha256"] = df["full_path"].map(sha256_file)
dups = df[df.duplicated("sha256", keep=False)].sort_values("sha256")
print("Rows involved in exact duplicates:", len(dups))
if len(dups):
    display(dups[["filepath", "label", "group_id", "sha256"]].head(30))
    # Exact duplicate copies must have the same label and group assignment.
    assert dups.groupby("sha256")["label"].nunique().max() == 1, "Conflicting labels for identical image bytes."
    assert dups.groupby("sha256")["group_id"].nunique().max() == 1, (
        "Identical image bytes have different group_id values. Fix the manifest."
    )
    df = df.drop_duplicates("sha256").reset_index(drop=True)
print("Dataset after exact duplicate removal:", len(df))

## 4. Group-disjoint train / validation / test split

In [ ]:
# Keep 20% of source groups for final test, then 20% of remaining groups for validation.
split1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
trainval_idx, test_idx = next(split1.split(df, df["label"], groups=df["group_id"]))
trainval_df = df.iloc[trainval_idx].copy()
test_df = df.iloc[test_idx].copy()

split2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED + 1)
train_idx, val_idx = next(split2.split(
    trainval_df, trainval_df["label"], groups=trainval_df["group_id"]
))
train_df = trainval_df.iloc[train_idx].copy()
val_df = trainval_df.iloc[val_idx].copy()

train_groups, val_groups, test_groups = map(
    set, [train_df.group_id, val_df.group_id, test_df.group_id]
)
assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

for name, part in [("TRAIN", train_df), ("VALIDATION", val_df), ("TEST", test_df)]:
    print(f"\n{name}: {len(part)} images, {part.group_id.nunique()} groups")
    print(part["label"].value_counts())
    assert set(part["label"].unique()) == {"FAKE", "REAL"}, (
        f"{name} does not contain both classes. Collect more groups per class "
        "or revise group assignments. Do not fall back to a random image split."
    )

split_manifest = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="validation"),
    test_df.assign(split="test")
])[["filepath", "label", "group_id", "split"]]
split_manifest.to_csv("split_manifest.csv", index=False)

## 5. Visual data inspection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["label"].value_counts().reindex(["FAKE", "REAL"]).plot(kind="bar", ax=axes[0], title="Image count by class")
df.groupby("label")["group_id"].nunique().reindex(["FAKE", "REAL"]).plot(kind="bar", ax=axes[1], title="Unique groups by class")
for ax in axes:
    ax.set_xlabel("Class")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
sample = df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(min(4, len(x)), random_state=SEED)
)
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    with Image.open(row["full_path"]) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(row["label"])
    ax.axis("off")
plt.tight_layout()

## 6. Image preprocessing and augmentation
Images are decoded as RGB, resized to 224×224 and scaled to `[0,1]`. Augmentation is applied only to training data. The serving API must use the same RGB resize and `[0,255]` input contract; the model itself performs rescaling.

In [ ]:
def decode_resize(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    return tf.cast(image, tf.float32), tf.cast(label, tf.float32)

def make_dataset(frame, training=False):
    paths = frame["full_path"].astype(str).values
    labels = frame["label"].map(CLASS_TO_ID).astype("int32").values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_raw = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.04),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomContrast(0.12),
], name="train_only_augmentation")

## 7. Class imbalance handling

In [ ]:
counts = train_df["label"].value_counts()
n = len(train_df)
class_weight = {
    CLASS_TO_ID[label]: n / (2.0 * count)
    for label, count in counts.items()
}
print("Class weights:", class_weight)

## 8. Build a custom CNN from scratch
This model uses randomly initialized convolutional layers, residual connections, batch normalization, dropout and global average pooling. There are **no pretrained weights** and no transfer learning.

In [ ]:
from tensorflow.keras import layers, Model

def conv_bn_relu(x, filters, kernel=3, stride=1):
    x = layers.Conv2D(filters, kernel, strides=stride, padding="same",
                      use_bias=False, kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)

def residual_block(x, filters, stride=1, dropout_rate=0.0):
    shortcut = x
    y = layers.Conv2D(filters, 3, strides=stride, padding="same",
                      use_bias=False, kernel_initializer="he_normal")(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False,
                      kernel_initializer="he_normal")(y)
    y = layers.BatchNormalization()(y)

    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding="same",
                                 use_bias=False, kernel_initializer="he_normal")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    y = layers.Add()([y, shortcut])
    y = layers.Activation("relu")(y)
    if dropout_rate:
        y = layers.SpatialDropout2D(dropout_rate)(y)
    return y

inputs = layers.Input(shape=(224, 224, 3), name="rgb_image")
x = layers.Rescaling(1.0 / 255.0, name="pixel_scaling")(inputs)
x = augmentation(x)
x = conv_bn_relu(x, 32, 3, 2)          # 112 x 112
x = residual_block(x, 32)
x = residual_block(x, 64, stride=2, dropout_rate=0.05)   # 56 x 56
x = residual_block(x, 64)
x = residual_block(x, 128, stride=2, dropout_rate=0.10)  # 28 x 28
x = residual_block(x, 128)
x = residual_block(x, 256, stride=2, dropout_rate=0.15)  # 14 x 14
x = residual_block(x, 256)
x = residual_block(x, 384, stride=2, dropout_rate=0.20)  # 7 x 7
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu", kernel_initializer="he_normal")(x)
x = layers.Dropout(0.45)(x)
outputs = layers.Dense(1, activation="sigmoid", name="p_real")(x)

model = Model(inputs, outputs, name="DeepSight_CustomCNN")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)
model.summary()

## 9. Train the CNN

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_deepsight_from_scratch.keras",
        monitor="val_auc", mode="max", save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", mode="max", patience=7, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.CSVLogger("training_log.csv")
]
history = model.fit(
    train_raw,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["loss", "auc"]):
    ax.plot(history.history[metric], label="train")
    ax.plot(history.history["val_" + metric], label="validation")
    ax.set_title(metric.upper())
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()

## 10. Select a decision threshold using validation data only
Do not select the threshold using the final test set. Here we choose the threshold that maximizes validation F1; for some applications you may instead prioritize recall or precision.

In [ ]:
def collect_predictions(dataset, trained_model):
    y_true, y_prob = [], []
    for xb, yb in dataset:
        y_true.extend(yb.numpy().ravel().astype(int))
        y_prob.extend(trained_model.predict(xb, verbose=0).ravel())
    return np.asarray(y_true), np.asarray(y_prob)

y_val, p_val = collect_predictions(val_ds, model)
precision, recall, thresholds = precision_recall_curve(y_val, p_val)
if len(thresholds):
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-8)
    best_idx = int(np.nanargmax(f1_values))
    BEST_THRESHOLD = float(thresholds[best_idx])
else:
    BEST_THRESHOLD = 0.5
print("Validation-selected threshold:", BEST_THRESHOLD)
print("Validation F1 at threshold:", f1_score(y_val, (p_val >= BEST_THRESHOLD).astype(int)))

## 11. Final independent test evaluation

In [ ]:
y_test, p_test = collect_predictions(test_ds, model)
y_pred = (p_test >= BEST_THRESHOLD).astype(int)
print(classification_report(
    y_test, y_pred, labels=[0,1], target_names=["FAKE","REAL"], zero_division=0
))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, p_test))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred, labels=[0,1]),
    display_labels=["FAKE","REAL"]
).plot()
plt.show()

## 12. JPEG compression stress test

In [ ]:
def compressed_dataset(frame, quality=40):
    def generator():
        for _, row in frame.iterrows():
            with Image.open(row["full_path"]) as im:
                im = im.convert("RGB").resize(IMAGE_SIZE)
                buf = BytesIO()
                im.save(buf, format="JPEG", quality=quality)
                buf.seek(0)
                arr = np.asarray(Image.open(buf).convert("RGB"), dtype=np.float32)
            yield arr, np.float32(CLASS_TO_ID[row["label"]])
    return tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec((224,224,3), tf.float32),
            tf.TensorSpec((), tf.float32)
        )
    ).batch(BATCH_SIZE).prefetch(AUTOTUNE)

compressed_test_ds = compressed_dataset(test_df, quality=40)
y_comp, p_comp = collect_predictions(compressed_test_ds, model)
pred_comp = (p_comp >= BEST_THRESHOLD).astype(int)
print("JPEG quality 40 stress-test report")
print(classification_report(y_comp, pred_comp, labels=[0,1], target_names=["FAKE","REAL"], zero_division=0))
print("Compressed ROC-AUC:", roc_auc_score(y_comp, p_comp))

## 13. Save the model and serving configuration
The Flask app expects RGB input `(224,224,3)` in `[0,255]`, sigmoid output `P(REAL)`. This notebook's model includes the `[0,1]` rescaling layer internally, so backend preprocessing must not divide by 255.

In [ ]:
model.save("deepsight_detector.keras")
config = {
    "model_name": "DeepSight_CustomCNN",
    "input_shape": [224,224,3],
    "input_range": [0,255],
    "output": "sigmoid_probability_real",
    "class_mapping": {"FAKE": 0, "REAL": 1},
    "decision_threshold": BEST_THRESHOLD,
    "pretrained_weights": False
}
Path("deepsight_model_config.json").write_text(json.dumps(config, indent=2))
print("Saved deepsight_detector.keras and deepsight_model_config.json")